# FB15k-237 — preprocessing

Turns the official FB15k-237 **triple split** and the **classes** of Jain et al. 2021 ("Do Embeddings Actually Capture Knowledge Graph Semantics?") into the files the embedding and classification work consumes, in `data/processed/fb15k-237/` (vocabulary: see `GLOSSARY.md`):

1. `train.tsv`, `valid.tsv`, `test.tsv`: the official triple split, `head<TAB>relation<TAB>tail`, no header, content and row order unchanged.
2. `entity_classes.tsv`: one row per (entity, class) pair, restricted to entities that occur in the triples. The classes are used as-is, without a WordNet closure; the closure is only *reported* in section 3.1.
3. `entity_split.tsv`: for every **experiment**, the 80/20 **entity split** of the entities that have at least one class of that experiment, following the rules of the authors' `classification.ipynb` (pinned commit `2d93782`).

Every claim about the outputs is checked by the acceptance assertions in section 6; the summary table in section 7 closes the notebook.

In [1]:
from pathlib import Path
import csv
import hashlib
import re
import shutil
import urllib.request

import nltk
import pandas as pd
from IPython.display import display
from nltk.corpus import wordnet as wn
from sklearn.model_selection import train_test_split

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

DATASET = 'fb15k-237'
search_dirs = [Path.cwd(), Path.cwd().parent]
REPO = next((p for p in search_dirs if (p / 'data' / 'fb15k-237_train.tsv').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Run from the repository root or the data/ directory.')
COMMITTED_TRAIN = REPO / 'data' / 'fb15k-237_train.tsv'
EXTERNAL_DIR = REPO / 'data' / 'external' / DATASET
OUT_DIR = REPO / 'data' / 'processed' / DATASET
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_CLASS_SIZE = 40
TEST_SIZE = 0.2
RANDOM_STATE = 37
SPLITS = ['train', 'valid', 'test']
TRIPLE_COLUMNS = ['head', 'relation', 'tail']
TRIPLE_EXT = 'txt'

# Pinned sources.
HF = 'https://huggingface.co/datasets/KGraph/FB15k-237/resolve/c7368ccc03358758270dbf9e475222444d19926b/data/'
AUTHORS = 'https://raw.githubusercontent.com/nitishajain/KGESemanticAnalysis/2d93782b55e169ec1f5e79c9c6245b709b44c86b/'
SOURCES = {  # file name -> (url, sha256)
    'train.txt': (HF + 'train.txt', '6e4c2782169af21e9743f3b1d200886f5d595bf6bc504ec1351720949c5cdfae'),
    'valid.txt': (HF + 'valid.txt', '749cbe9d923bac7b9354da5614ecfed2e0220256d442c3e04a6b303db1f273d9'),
    'test.txt': (HF + 'test.txt', 'e2e35e8e6113de220140b6f44dc71a5207b0fc6872d575e874aefe13259b655b'),
    'freebaseTypes.tsv': (AUTHORS + 'data/freebase/freebaseTypes.tsv',
                          'd21acba61828b8a773860047be2fd23692accc36b10001eaf5c40a401e57cb30'),
}
EXPECTED_TRIPLES = {'train': 272_115, 'valid': 17_535, 'test': 20_466}

# Experiments, verbatim from the authors' classification.ipynb (class IDs in YAGO form).
EXPERIMENTS = {
    'Level-1': ['wordnet_person_100007846', 'wordnet_organization_108008335',
                'wordnet_body_of_water_109225146', 'wordnet_product_104007894'],
    'Level-2-Organizations': ['wordnet_musical_organization_108246613', 'wordnet_party_108256968',
                              'wordnet_enterprise_108056231', 'wordnet_nongovernmental_organization_108009834'],
    'Level-2-Persons': ['wordnet_artist_109812338', 'wordnet_officeholder_110371450', 'wordnet_writer_110794014',
                        'wordnet_scientist_110560637', 'wordnet_politician_110450303'],
    'Level-3-Artists': ['wordnet_painter_110391653', 'wordnet_sculptor_110566072',
                        'wordnet_photographer_110426749', 'wordnet_illustrator_109812068',
                        'wordnet_printmaker_110475687'],
}
EXPERIMENT_CLASSES = sorted({c for classes in EXPERIMENTS.values() for c in classes})


def read_triples(path):
    return read_tsv(path, header=False).set_axis(TRIPLE_COLUMNS, axis=1)



def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def read_tsv(path, header=True):
    # QUOTE_NONE: the TSVs we write are never quoted, quote characters inside names are literal.
    return pd.read_csv(path, sep='\t', quoting=csv.QUOTE_NONE, dtype='string', keep_default_na=False,
                       encoding='utf-8', header=0 if header else None)


def write_tsv(df, path, header=True):
    # Written by hand: UTF-8, LF line endings, never quoted. Values must not contain control separators.
    for col in df.columns:
        assert not df[col].str.contains('[\t\n\r]', regex=True).any(), f'tab/newline inside column {col!r}'
    lines = df.iloc[:, 0]
    for col in df.columns[1:]:
        lines = lines + '\t' + df[col]
    text = lines.str.cat(sep='\n') + '\n'
    if header:
        text = '\t'.join(df.columns) + '\n' + text
    with open(path, 'w', encoding='utf-8', newline='\n') as f:
        f.write(text)

## 1. Download and verify

Every source is pinned to a commit and its SHA256 is asserted: the triple split from Hugging Face and the classes from the authors' repository (Jain et al. 2021). Downloads go to `data/external/fb15k-237/` (gitignored). The SHA256 of the authors' class file was recorded from the pinned commit of their repository (they do not publish one); the SHA256 values of the three triple files are those of the pinned Hugging Face commit. In the pinned Hugging Face commit the train file has CRLF line endings, valid and test have LF; the processed files are all written with LF.

In [2]:
def fetch(name, url, expected):
    dest = EXTERNAL_DIR / name
    if not (dest.is_file() and sha256(dest) == expected):
        dest.parent.mkdir(parents=True, exist_ok=True)
        request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(request, timeout=120) as response, open(dest, 'wb') as f:
            shutil.copyfileobj(response, f)
    assert sha256(dest) == expected, f'SHA256 mismatch for {name}'
    return dest


downloads = {name: fetch(name, url, digest) for name, (url, digest) in SOURCES.items()}
pd.Series({name: f'{path.stat().st_size:,} bytes' for name, path in downloads.items()}, name='downloaded')

train.txt            21,277,292 bytes
valid.txt             1,285,566 bytes
test.txt              1,499,033 bytes
freebaseTypes.tsv     7,980,767 bytes
Name: downloaded, dtype: str

## 2. Triple split

The official split is copied unchanged: no deduplication, no self-loop removal, no relation merging, same row order. Only the format changes (line terminator normalised to LF). The downloaded train split must hold the same triples as the train file committed to this repository (compared by content, since `core.autocrlf` makes working-tree bytes differ).

In [3]:
triples = {split: read_triples(EXTERNAL_DIR / f'{split}.{TRIPLE_EXT}') for split in SPLITS}
for split, df in triples.items():
    assert list(df.columns) == TRIPLE_COLUMNS, (split, list(df.columns))
    assert len(df) == EXPECTED_TRIPLES[split], (split, len(df))

In [4]:
committed = read_triples(COMMITTED_TRAIN)
assert committed.equals(triples['train']), 'downloaded train split differs from the committed train file'

for split, df in triples.items():
    write_tsv(df, OUT_DIR / f'{split}.tsv', header=False)

entities = sorted(set().union(*(set(df['head']) | set(df['tail']) for df in triples.values())))
pd.DataFrame({'triples': {s: len(df) for s, df in triples.items()},
              'self_loops': {s: int((df['head'] == df['tail']).sum()) for s, df in triples.items()},
              'duplicate_rows': {s: int(df.duplicated().sum()) for s, df in triples.items()}}).assign(
    entities_in_dataset=len(entities))

,triples,self_loops,duplicate_rows,entities_in_dataset
train,272115,1625,0,14541
valid,17535,90,0,14541
test,20466,91,0,14541


## 3. Entity classes

`freebaseTypes.tsv` maps Freebase entities to WordNet classes (`/m/xxx<TAB>wordnet_class_id`, no header). It is used **as-is**; only entities that occur in the triples are kept, written sorted and deduplicated to `entity_classes.tsv`.

### 3.1 Read and write

In [5]:
class_rows = read_tsv(EXTERNAL_DIR / 'freebaseTypes.tsv', header=False).set_axis(['entity', 'class'], axis=1)
entity_set = set(entities)
in_dataset = class_rows['entity'].isin(entity_set)
entity_classes = (class_rows[in_dataset].drop_duplicates().sort_values(['entity', 'class'])
                  .reset_index(drop=True))
write_tsv(entity_classes, OUT_DIR / 'entity_classes.tsv', header=True)
pd.Series({
    'rows in freebaseTypes.tsv': f'{len(class_rows):,}',
    'rows whose entity is not in the triples (dropped)': f'{int((~in_dataset).sum()):,}',
    'entity-class pairs kept': f'{len(entity_classes):,}',
    'dataset entities with >= 1 class': f'{entity_classes["entity"].nunique():,}',
    'dataset entities total': f'{len(entities):,}',
    'coverage': f'{entity_classes["entity"].nunique() / len(entities):.1%}',
    'distinct classes': f'{entity_classes["class"].nunique():,}',
}, name='value').to_frame()

,value
rows in freebaseTypes.tsv,"212,270"
rows whose entity is not in the triples (dropped),0
entity-class pairs kept,"212,270"
dataset entities with >= 1 class,"12,615"
dataset entities total,"14,541"
coverage,86.8%
distinct classes,"2,260"


### 3.2 What a WordNet closure would add (reported, not applied)

The classes are not guaranteed to be transitively closed. For information only: map each class ID to its WordNet 3.0 synset by offset (`wordnet_<lemma>_1<8-digit offset>`), take all ancestors over `hypernyms() + instance_hypernyms()`, and count how many (entity, experiment class) memberships that would add. Experiment classes are matched by offset.

In [6]:
nltk.download('wordnet', quiet=True)
assert wn.get_version() == '3.0', wn.get_version()


def offset_of(class_id):
    m = re.fullmatch(r'wordnet_.+_1(\d{8})', class_id)
    assert m, f'not a YAGO WordNet class ID: {class_id}'
    return int(m.group(1))


def synset_at(offset):
    return wn.synset_from_pos_and_offset('n', offset)


def parents(synset):
    return synset.hypernyms() + synset.instance_hypernyms()

In [7]:
class_offset = {c: offset_of(c) for c in entity_classes['class'].unique()}
experiment_offset = {c: offset_of(c) for c in EXPERIMENT_CLASSES}
ancestors = {off: {off} | {s.offset() for s in synset_at(off).closure(parents)}
             for off in sorted(set(class_offset.values()) | set(experiment_offset.values()))}
print(f'{len(class_offset):,} distinct classes expand to {len(set().union(*ancestors.values())):,} distinct WordNet synsets')

entity_offsets = {}
for e, c in zip(entity_classes['entity'], entity_classes['class']):
    entity_offsets.setdefault(e, set()).add(class_offset[c])
closure_rows = []
for experiment, classes in EXPERIMENTS.items():
    for c in classes:
        off = experiment_offset[c]
        raw_members = {e for e, offs in entity_offsets.items() if off in offs}
        closed_members = {e for e, offs in entity_offsets.items() if any(off in ancestors[o] for o in offs)}
        closure_rows.append({'experiment': experiment, 'class': c, 'raw': len(raw_members),
                             'with closure': len(closed_members), 'would add': len(closed_members) - len(raw_members),
                             'kept raw (>=40)': len(raw_members) >= MIN_CLASS_SIZE,
                             'kept with closure': len(closed_members) >= MIN_CLASS_SIZE})
closure_report = pd.DataFrame(closure_rows)
display(closure_report)
print('memberships a closure would add to the experiment classes:', int(closure_report['would add'].sum()))

2,260 distinct classes expand to 2,262 distinct WordNet synsets


,experiment,class,raw,with closure,would add,kept raw (>=40),kept with closure
0,Level-1,wordnet_person_100007846,4446,4446,0,True,True
1,Level-1,wordnet_organization_108008335,2360,2360,0,True,True
2,Level-1,wordnet_body_of_water_109225146,24,24,0,False,False
3,Level-1,wordnet_product_104007894,1593,1593,0,True,True
4,Level-2-Organizations,wordnet_musical_organization_108246613,137,137,0,True,True
5,Level-2-Organizations,wordnet_party_108256968,42,42,0,True,True
6,Level-2-Organizations,wordnet_enterprise_108056231,170,170,0,True,True
7,Level-2-Organizations,wordnet_nongovernmental_organization_108009834,82,82,0,True,True
8,Level-2-Persons,wordnet_artist_109812338,1892,1892,0,True,True
9,Level-2-Persons,wordnet_officeholder_110371450,108,108,0,True,True


memberships a closure would add to the experiment classes: 0


## 4. Entity split

For every experiment, as in the authors' `classification.ipynb`: candidates are all dataset entities sorted by name; classes with fewer than 40 members are dropped; entities with at least one remaining experiment class are kept; then `train_test_split(entities, test_size=0.2, random_state=37)`, not stratified. `entity_split.tsv` stores, per kept entity, its split and its remaining experiment classes (sorted, comma-separated).

In [8]:
members = {c: set(g['entity']) for c, g in entity_classes[entity_classes['class'].isin(EXPERIMENT_CLASSES)].groupby('class')}

split_rows, experiment_rows, class_summary_rows = [], [], []
for experiment, classes in EXPERIMENTS.items():
    counts = {c: len(members.get(c, ())) for c in classes}
    kept = [c for c in classes if counts[c] >= MIN_CLASS_SIZE]
    dropped = [c for c in classes if counts[c] < MIN_CLASS_SIZE]
    entity_to_classes = {}
    for c in kept:
        for e in members[c]:
            entity_to_classes.setdefault(e, []).append(c)
    split_entities = sorted(e for e in entities if e in entity_to_classes)
    train_entities, test_entities = train_test_split(split_entities, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    for split, group in (('train', train_entities), ('test', test_entities)):
        for e in sorted(group):
            split_rows.append((experiment, e, split, ','.join(sorted(entity_to_classes[e]))))
    experiment_rows.append({'experiment': experiment, 'classes kept': len(kept), 'classes dropped': len(dropped),
                            'dropped': ', '.join(dropped), 'entities': len(split_entities),
                            'train': len(train_entities), 'test': len(test_entities)})
    class_summary_rows += [{'experiment': experiment, 'class': c, 'members': counts[c], 'kept': c in kept}
                           for c in classes]

entity_split = pd.DataFrame(split_rows, columns=['experiment', 'entity', 'split', 'classes']).astype('string')
write_tsv(entity_split, OUT_DIR / 'entity_split.tsv', header=True)
experiment_summary = pd.DataFrame(experiment_rows)
class_summary = pd.DataFrame(class_summary_rows)
experiment_summary

,experiment,classes kept,classes dropped,dropped,entities,train,test
0,Level-1,3,1,wordnet_body_of_water_109225146,8399,6719,1680
1,Level-2-Organizations,4,0,,428,342,86
2,Level-2-Persons,5,0,,3172,2537,635
3,Level-3-Artists,2,3,"wordnet_sculptor_110566072, wordnet_illustrator_109812068, wordnet_printmaker_110475687",170,136,34


## 5. Outputs written

Files in `data/processed/fb15k-237/`. Section 6 re-reads them from disk.

In [9]:
for p in sorted(OUT_DIR.iterdir()):
    print(f'{p.name:20s} {p.stat().st_size:>10,d} bytes')

entity_classes.tsv    7,980,780 bytes
entity_split.tsv        676,367 bytes
test.tsv              1,499,033 bytes
train.tsv            21,005,177 bytes
valid.tsv             1,285,566 bytes


## 6. Acceptance checks

These cells read the files on disk and recompute their expectations without reusing the processing code above, so they can disagree with it. Expected counts are literals from the spec. A failing assertion stops the notebook.

In [10]:
# 6.1 Downloads, triple files and entity universe
def read_rows(path):
    with open(path, encoding='utf-8', newline='') as f:
        text = f.read()
    return text.replace('\r\n', '\n').rstrip('\n').split('\n')

for name, (_, digest) in SOURCES.items():
    assert sha256(EXTERNAL_DIR / name) == digest, f'{name}: SHA256 mismatch'

triple_entities = set()
for split in SPLITS:
    raw = (OUT_DIR / f'{split}.tsv').read_bytes()
    assert b'\r' not in raw and raw.endswith(b'\n'), f'{split}.tsv must be LF-terminated'
    raw.decode('utf-8')
    triple_rows = [line.split('\t') for line in read_rows(OUT_DIR / f'{split}.tsv')]
    assert len(triple_rows) == EXPECTED_TRIPLES[split], (split, len(triple_rows))
    assert all(len(r) == 3 for r in triple_rows), f'{split}.tsv must have exactly 3 columns, no header'
    # unchanged content and row order versus the download
    assert triple_rows == [line.split('\t') for line in read_rows(EXTERNAL_DIR / f'{split}.{TRIPLE_EXT}')]
    for head, _, tail in triple_rows:
        triple_entities.update((head, tail))
    if split == 'train':
        committed_rows = [line.split('\t') for line in read_rows(COMMITTED_TRAIN)]
        assert triple_rows == committed_rows, 'download differs from committed data/fb15k-237_train.tsv'
assert len(triple_entities) == 14_541, len(triple_entities)
print('triple files OK;', len(triple_entities), 'entities')

triple files OK; 14541 entities


In [11]:
# 6.2 entity_classes.tsv
raw = (OUT_DIR / 'entity_classes.tsv').read_bytes()
assert b'\r' not in raw and raw.endswith(b'\n')
lines = raw.decode('utf-8').rstrip('\n').split('\n')
assert lines[0] == 'entity\tclass'
pairs = [tuple(l.split('\t')) for l in lines[1:]]
assert all(len(p) == 2 for p in pairs)
assert pairs == sorted(pairs) and len(pairs) == len(set(pairs)), 'must be sorted, one row per pair'
assert {e for e, _ in pairs} <= triple_entities, 'entity_classes has entities outside the triples'
assert not any(c.startswith('<') or c.endswith('>') for _, c in pairs), 'class IDs carry angle brackets'
class_members = {}
for e, c in pairs:
    class_members.setdefault(c, set()).add(e)
print(len(pairs), 'pairs,', len({e for e, _ in pairs}), 'entities with a class,', len(class_members), 'classes')

212270

 pairs, 12615 entities with a class, 2260 classes


In [12]:
# 6.3 entity_split.tsv, per experiment
raw = (OUT_DIR / 'entity_split.tsv').read_bytes()
assert b'\r' not in raw and raw.endswith(b'\n')
lines = raw.decode('utf-8').rstrip('\n').split('\n')
assert lines[0] == 'experiment\tentity\tsplit\tclasses'
file_rows = [l.split('\t') for l in lines[1:]]
assert all(len(r) == 4 for r in file_rows)
assert {r[0] for r in file_rows} == set(EXPERIMENTS), 'experiments must be exactly the four of the spec'

for experiment, classes in EXPERIMENTS.items():
    group_rows = [r for r in file_rows if r[0] == experiment]
    group_entities = [r[1] for r in group_rows]
    assert len(group_entities) == len(set(group_entities)), f'{experiment}: duplicate entities'
    assert set(group_entities) <= triple_entities
    train = {r[1] for r in group_rows if r[2] == 'train'}
    test = {r[1] for r in group_rows if r[2] == 'test'}
    assert {r[2] for r in group_rows} <= {'train', 'test'}
    assert not train & test, f'{experiment}: train/test overlap'
    n = len(group_rows)
    assert len(test) == (n + 4) // 5, f'{experiment}: |test| must be ceil(0.2 n)'  # ceil(n / 5) in integers
    # classes kept = experiment classes with >= 40 member entities, counted from entity_classes.tsv
    kept = sorted(c for c in classes if len(class_members.get(c, ())) >= MIN_CLASS_SIZE)
    assert kept, f'{experiment}: no class survives'
    # every entity with a kept class is in the split, with exactly its kept classes
    expected = {}
    for c in kept:
        for e in class_members[c]:
            expected.setdefault(e, []).append(c)
    assert {r[1]: r[3] for r in group_rows} == {e: ','.join(sorted(cs)) for e, cs in expected.items()}, \
        f'{experiment}: classes column disagrees with entity_classes.tsv'
    assert all(r[3] for r in group_rows), f'{experiment}: row without a class'
    per_class = {c: sum(c in r[3].split(',') for r in group_rows) for c in kept}
    assert all(v >= MIN_CLASS_SIZE for v in per_class.values()), f'{experiment}: class below {MIN_CLASS_SIZE}'
    # the sample assignment: sorted candidates, same random state, not stratified
    tr, te = train_test_split(sorted(group_entities), test_size=0.2, random_state=37)
    assert set(tr) == train and set(te) == test, f'{experiment}: split differs from train_test_split'
    print(f'{experiment}: OK ({n} entities, {len(train)}/{len(test)} train/test, kept {len(kept)}/{len(classes)} classes)')

Level-1: OK (8399 entities, 6719/1680 train/test, kept 3/4 classes)
Level-2-Organizations: OK (428 entities, 342/86 train/test, kept 4/4 classes)
Level-2-Persons: OK (3172 entities, 2537/635 train/test, kept 5/5 classes)
Level-3-Artists: OK (170 entities, 136/34 train/test, kept 2/5 classes)


In [13]:
# 6.4 The files re-read from disk equal the DataFrames that were written (same dtypes, same row order)
for split in SPLITS:
    written_triples = read_tsv(OUT_DIR / f'{split}.tsv', header=False).set_axis(TRIPLE_COLUMNS, axis=1)
    pd.testing.assert_frame_equal(written_triples, triples[split])
pd.testing.assert_frame_equal(read_tsv(OUT_DIR / 'entity_classes.tsv'), entity_classes)
pd.testing.assert_frame_equal(read_tsv(OUT_DIR / 'entity_split.tsv'), entity_split)
print('re-read triple files, entity_classes.tsv and entity_split.tsv equal what was written')

re-read triple files, entity_classes.tsv and entity_split.tsv equal what was written


## 7. Summary

Per experiment: classes kept and dropped (fewer than 40 members), entities in the entity split and its train/test sizes; then the members of every experiment class among the dataset entities.

In [14]:
display(experiment_summary)
display(class_summary.set_index(['experiment', 'class']))

,experiment,classes kept,classes dropped,dropped,entities,train,test
0,Level-1,3,1,wordnet_body_of_water_109225146,8399,6719,1680
1,Level-2-Organizations,4,0,,428,342,86
2,Level-2-Persons,5,0,,3172,2537,635
3,Level-3-Artists,2,3,"wordnet_sculptor_110566072, wordnet_illustrator_109812068, wordnet_printmaker_110475687",170,136,34


members   kept
experiment            class                                                         
Level-1               wordnet_person_100007846                           4446   True
                      wordnet_organization_108008335                     2360   True
                      wordnet_body_of_water_109225146                      24  False
                      wordnet_product_104007894                          1593   True
Level-2-Organizations wordnet_musical_organization_108246613              137   True
                      wordnet_party_108256968                              42   True
                      wordnet_enterprise_108056231                        170   True
                      wordnet_nongovernmental_organization_108009834       82   True
Level-2-Persons       wordnet_artist_109812338                           1892   True
                      wordnet_officeholder_110371450                      108   True
                      wordnet_writer_110794014                           1919   True
                      wordnet_scientist_110560637                         115   True
                      wordnet_politician_110450303                        245   True
Level-3-Artists       wordnet_painter_110391653                            69   True
                      wordnet_sculptor_110566072                           15  False
                      wordnet_photographer_110426749                      110   True
                      wordnet_illustrator_109812068                        20  False
                      wordnet_printmaker_110475687                         11  False